In [22]:
# pip install pandas openpyxl torch transformers sentencepiece uuid
# import nltk
# nltk.download("wordnet")

[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/erinbuchanan/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

# Translate Word Pairs Function

For word pairs that have not previously been translated, we will first translate those pairs with a newer model than used in the original SPAML.

In [28]:
# clean words 
def clean_word(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)   # remove punctuation
    text = text.strip()
    return text

# translate the english words into the target language
def translate_pairs(df, target_lang, src_lang="eng", batch_size=64):

    """
    Translate cue-target pairs and fix cases where cue == target
    using WordNet synonyms.

    Requires global:
        processor
        model
    """

    # ------------------------------
    # 1 Translate all words
    # ------------------------------

    words = df["en_cue"].tolist() + df["en_target"].tolist()

    translations = []

    for i in range(0, len(words), batch_size):

        batch = words[i:i+batch_size]

        inputs = processor(
            text=batch,
            src_lang=src_lang,
            return_tensors="pt",
            padding=True
        )

        outputs = model.generate(
            **inputs,
            tgt_lang=target_lang,
            max_new_tokens=20
        )

        decoded = processor.batch_decode(
            outputs,
            skip_special_tokens=True
        )

        translations.extend(decoded)

    half = len(df)

    df[f"{target_lang}_cue"] = translations[:half]
    df[f"{target_lang}_target"] = translations[half:]

    # ------------------------------
    # 2 lowercase
    # ------------------------------

    # clean translations
    df[f"{target_lang}_cue"] = df[f"{target_lang}_cue"].apply(clean_word)
    df[f"{target_lang}_target"] = df[f"{target_lang}_target"].apply(clean_word)

    # ------------------------------
    # 3 detect cue == target
    # ------------------------------

    df["needs_review"] = df[f"{target_lang}_cue"] == df[f"{target_lang}_target"]

    df[f"suggested_{target_lang}_cue"] = None

    # ------------------------------
    # 4 generate synonym-based cue suggestions
    # ------------------------------

    for idx in df[df["needs_review"]].index:

        english_cue = df.loc[idx, "en_cue"]
        target_word = df.loc[idx, f"{target_lang}_target"]

        # Get WordNet synonyms
        synsets = wordnet.synsets(english_cue)

        synonyms = set()

        for syn in synsets:
            for lemma in syn.lemmas():
                word = lemma.name().replace("_", " ")
                if word.lower() != english_cue.lower():
                    synonyms.add(word)

        synonyms = list(synonyms)

        suggestion = None

        if synonyms:

            inputs = processor(
                text=synonyms,
                src_lang=src_lang,
                return_tensors="pt",
                padding=True
            )

            outputs = model.generate(
                **inputs,
                tgt_lang=target_lang,
                max_new_tokens=20
            )

            translated_syns = processor.batch_decode(
                outputs,
                skip_special_tokens=True
            )

            translated_syns = [clean_word(w) for w in translated_syns]

            for candidate in translated_syns:
                if candidate != target_word:
                    suggestion = candidate
                    break

        df.loc[idx, f"suggested_{target_lang}_cue"] = suggestion

    # ------------------------------
    # 5 create final cue column
    # ------------------------------

    df[f"final_{target_lang}_cue"] = df[f"{target_lang}_cue"]

    mask = df["needs_review"] & df[f"suggested_{target_lang}_cue"].notna()

    df.loc[mask, f"final_{target_lang}_cue"] = df.loc[
        mask, f"suggested_{target_lang}_cue"
    ]

    # ------------------------------
    # 6 save results
    # ------------------------------

    Path(target_lang).mkdir(exist_ok=True)

    df.to_csv(f"{target_lang}/{target_lang}_word_pairs.csv", index=False)

    print(f"Saved translated pairs to {target_lang}/{target_lang}_word_pairs.csv")
    print(f"{df['needs_review'].sum()} cue==target collisions detected")

    return df

# Create Randomized Lists

For the non-priming tasks, we need to create randomized lists of 400 words for the word pairs we need to norm. 

In [34]:
def build_word_lists(csv_path, lang, words_per_file=400, seed=42):

    random.seed(seed)

    df = pd.read_csv(csv_path)

    cue_col = f"final_{lang}_cue"
    target_col = f"{lang}_target"

    if cue_col not in df.columns or target_col not in df.columns:
        raise ValueError(f"{cue_col} or {target_col} not found")

    # Combine stimuli
    words = df[cue_col].tolist() + df[target_col].tolist()

    print(f"{len(words)} total stimuli")
    print(f"{len(set(words))} unique stimuli")

    random.shuffle(words)

    word_lists = []
    current_list = []

    for word in words:

        if word not in current_list:
            current_list.append(word)

        if len(current_list) == words_per_file:
            word_lists.append(current_list)
            current_list = []

    # Drop incomplete final list
    if len(current_list) == words_per_file:
        word_lists.append(current_list)

    # Ensure language folder exists
    Path(lang).mkdir(exist_ok=True)

    # Save lists
    for i, lst in enumerate(word_lists, start=1):

        pd.DataFrame({
            "position": range(1, len(lst)+1),
            "word": lst
        }).to_csv(
            f"{lang}/{lang}_wordlist_{i}.csv",
            index=False
        )

    print(f"{len(word_lists)} word lists created for {lang}")

    return word_lists

# Put into Tasks

Take the shuffled word lists and build different versions of the html tasks.

In [3]:
def insert_words_into_html(template_path, word_list, output_path):
    """
    Insert a list of words into the sampleWords array of an HTML template.
    """

    # Read template HTML
    with open(template_path, "r", encoding="utf-8") as f:
        html = f.read()

    # Convert word list into JS array format
    words_js = json.dumps(word_list, ensure_ascii=False, indent=2)

    new_block = f"const sampleWords = {words_js};"

    # Replace the sampleWords block
    import re
    html = re.sub(
        r"const sampleWords\s*=\s*\[[\s\S]*?\];",
        new_block,
        html
    )

    # Write new HTML
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(html)

    print(f"Created {output_path}")

# Build the HTML files

This part will build the html files with the word lists and translate the words for the experiment. 

In [4]:
def generate_html_tasks(
    template_dir,
    output_dir,
    translation_csv,
    word_lists,
    lang,
    save_wordlists=True
):

    translations = pd.read_csv(translation_csv)

    Path(output_dir).mkdir(parents=True, exist_ok=True)

    templates = sorted(Path(template_dir).glob("*.html"))

    # SAVE WORD LISTS
    if save_wordlists:
        stimuli_dir = Path(output_dir) / "stimuli"
        stimuli_dir.mkdir(exist_ok=True)

        for i, words in enumerate(word_lists, start=1):
            pd.DataFrame({"word": words}).to_csv(
                stimuli_dir / f"wordlist_{lang}_{i}.csv",
                index=False
            )

    for template in templates:

        with open(template, "r", encoding="utf-8") as f:
            html = f.read()

        # Replace UI strings
        for _, row in translations.iterrows():
            english = str(row["english"])
            translated = str(row[f"{lang}"])
            html = html.replace(english, translated)

        # Generate HTML with word lists
        for i, word_list in enumerate(word_lists, start=1):

            words_js = json.dumps(word_list, ensure_ascii=False, indent=2)

            new_block = f"const sampleWords = {words_js};"

            html_with_words = re.sub(
                r"const sampleWords\s*=\s*\[[\s\S]*?\];",
                new_block,
                html
            )

            outfile = (
                Path(output_dir)
                / f"{template.stem}_{lang}_{i}.html"
            )

            with open(outfile, "w", encoding="utf-8") as f:
                f.write(html_with_words)

            # Save translation transparency file
            translation_snapshot = translations[["id", "english", lang]]
            
            snapshot_path = Path(output_dir) / f"translation_map_{lang}.csv"
            
            translation_snapshot.to_csv(snapshot_path, index=False)

            print("Created", outfile)


# Build .jas Files

These files are required to make a valid jzip for JATOS.

In [5]:
def generate_jas(template_path, output_path, experiment, lang, version):

    with open(template_path, "r") as f:
        jas = json.load(f)

    name = f"{experiment}_{lang}_{version}"

    # new UUIDs
    study_uuid = str(uuid.uuid4())
    component_uuid = str(uuid.uuid4())
    batch_uuid = str(uuid.uuid4())

    jas["data"]["uuid"] = study_uuid
    jas["data"]["title"] = name
    jas["data"]["description"] = f"{experiment} study ({lang}) version {version}"
    jas["data"]["dirName"] = name

    jas["data"]["componentList"][0]["uuid"] = component_uuid
    jas["data"]["componentList"][0]["title"] = name

    jas["data"]["batchList"][0]["uuid"] = batch_uuid

    with open(output_path, "w") as f:
        json.dump(jas, f, indent=2)

    print("Created", output_path)

# Libraries and Models

In [30]:
import pandas as pd
from transformers import AutoProcessor, AutoModelForSeq2SeqLM
import random
from pathlib import Path
import json
import uuid
from pathlib import Path
from nltk.corpus import wordnet
import re

# Load model once (important for speed)
model_name = "facebook/seamless-m4t-v2-large"

processor = AutoProcessor.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

word_pairs_english = pd.read_csv("en_words.csv")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

## Afrikaans

In [33]:
# first translate the words
# translate_pairs(word_pairs_english, "afr")

# second manually edit files save as _update
word_lists = build_word_lists("afr/afr_word_pairs_update.csv", "afr")


2000 total stimuli
1915 unique stimuli
5 word lists created for afr
